# Intermediate Model Analysis
**BRIM Systems** | Prepared by: Brian Davis

---

## Purpose

Exploratory analysis against dbt intermediate model output. Reads from the enriched
fact tables produced by the intermediate layer — where all five source systems are
joined for the first time — to surface cross-system patterns, validate analytical
assumptions, and inform mart model and dashboard design.

**Section flow:**
1. Configuration & Data Loading
2. Intermediate Table Summary (preview, schema/nulls, grain verification)
3. Cross-System Correlation Matrix
4. Additional Analyses (Defect Rate Analysis, Scrap Cost Attribution, Operator & Certification Analysis)

## 0. Environment Setup

In [1]:
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import linregress, ttest_ind

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

BRAND_BLUE   = "#3D5166"
BRAND_ACCENT = "#6B8FA8"
WARN_AMBER   = "#D4881E"
FAIL_RED     = "#B94040"
PASS_GREEN   = "#4A7C59"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#CCCCCC", "axes.grid": True,
    "grid.color": "#EEEEEE", "grid.linestyle": "-",
    "font.family": "sans-serif", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "xtick.labelsize": 10,
    "ytick.labelsize": 10, "legend.fontsize": 10,
    "figure.dpi": 120,
})

print("Environment ready.")

Environment ready.


## 1. Configuration & Data Loading

In [2]:
# ── Database connection ────────────────────────────────────────────────────
DB_PATH = Path("../data/c01.duckdb").resolve()

# ── Intermediate table name constants ─────────────────────────────────────
TBL_INT_ORDERS  = "int_quality__orders_enriched"
TBL_INT_SCRAP     = "int_quality__scrap_costs"

# ── Primary keys ───────────────────────────────────────────────────────────
PK_MAP = {
    TBL_INT_ORDERS: "work_order_id",
    TBL_INT_SCRAP:    "scrap_id",
}

In [ ]:
# ── Load tables ───────────────────────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))
dfs = {}

print(f"{'Table':<45} {'Rows':>8}  {'Cols':>5}")
print("-" * 65)

for tbl in [TBL_INT_ORDERS, TBL_INT_SCRAP]:
    df = con.execute(f"SELECT * FROM {tbl}").df()
    dfs[tbl] = df
    print(f"  ✓  {tbl:<42} {len(df):>8,}  {len(df.columns):>5}")

orders = dfs[TBL_INT_ORDERS]
scrap    = dfs[TBL_INT_SCRAP]

print("\nTables loaded successfully.")

Table                                             Rows   Cols
-----------------------------------------------------------------


IOException: IO Error: No files found that match the pattern "/workspaces/portfolio/cases/c01_quality_scrap/data/raw/erp/production_orders.csv"

: 

## 2. Intermediate Table Summary

### 2.1 Preview

In [ ]:
for tbl, df in dfs.items():
    print(f"\n{'='*70}")
    print(f"  {tbl.upper()}  ({len(df):,} rows  x  {len(df.columns)} columns)")
    print(f"{'='*70}")
    display(df.head())

### 2.2 Schema & Null Flags

In [ ]:
# ── Null thresholds ────────────────────────────────────────────────────────
NULL_WARN_PCT = 0
NULL_FAIL_PCT = 10.0

In [ ]:
def schema_summary(df):
    rows = []
    for col in df.columns:
        s        = df[col]
        null_ct  = s.isna().sum()
        null_pct = null_ct / len(df) * 100
        rows.append({
            "column":     col,
            "dtype":      str(s.dtype),
            "null_count": null_ct,
            "null_pct":   round(null_pct, 1),
            "n_unique":   s.nunique(dropna=True),
            "sample":     str(s.dropna().iloc[0])[:60] if null_ct < len(df) else "(all null)",
        })
    return pd.DataFrame(rows)

for tbl, df in dfs.items():
    print(f"\n{'='*70}\n  {tbl.upper()}  ({len(df):,} rows, {len(df.columns)} columns)\n{'='*70}")
    display(
        schema_summary(df).style
        .format({"null_pct": "{:.1f}%"})
        .applymap(
            lambda v: "background-color: #F8D7DA" if isinstance(v, float) and v > NULL_FAIL_PCT else
                      "background-color: #FFF3CD" if isinstance(v, float) and v > NULL_WARN_PCT else "",
            subset=["null_pct"]
        )
        .set_properties(**{"font-size": "11px"})
    )

### 2.3 Grain Verification

Confirms the join logic in the intermediate models did not introduce fanout.
Each model should retain exactly one row per its declared grain.

In [ ]:
print(f"{'Table':<45} {'PK':<22} {'Total':>8}  {'Unique':>8}  {'Dupes':>8}  Status")
print("-" * 100)

for tbl, pk in PK_MAP.items():
    df      = dfs[tbl]
    total   = len(df)
    unique  = df[pk].nunique()
    dupes   = total - unique
    status  = "✓ PASS" if dupes == 0 else "✗ FAIL — join introduced fanout"
    print(f"  {status[:1]}  {tbl:<42} {pk:<22} {total:>8,}  {unique:>8,}  {dupes:>8,}  {status}")

---
## 3. Cross-System Correlation Matrix

In [ ]:
# ── Columns to include in correlation matrix ──────────────────────────────
# Boolean and binary columns are cast to int (0/1) so they appear in the matrix.
# Non-numeric and high-cardinality ID columns are excluded.
CORR_EXCLUDE = [
    "work_order_id", "inspection_id", "part_number", "customer",
    "machine_id", "operator_id", "lot_id", "inspector_id",
    "machine_name", "operator_name", "machine_location",
    "defect_code", "disposition", "material_type",
    "shift_code", "complexity", "supplier", "lot_cert_status",
    "specialization", "cert_level", "operator_home_shift",
]

corr_df = orders.copy()

# Cast boolean columns to int
bool_cols = corr_df.select_dtypes(include="bool").columns.tolist()
for col in bool_cols:
    corr_df[col] = corr_df[col].astype(int)

# Drop excluded and non-numeric columns
corr_df = corr_df.drop(columns=[c for c in CORR_EXCLUDE if c in corr_df.columns], errors="ignore")
corr_df = corr_df.select_dtypes(include="number")

print(f"Columns included in correlation matrix ({len(corr_df.columns)}):")
print(corr_df.columns.tolist())

In [ ]:
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(max(8, len(corr_df.columns) * 1.1),
                                max(6, len(corr_df.columns) * 0.9)))
sns.heatmap(
    corr_matrix, ax=ax,
    annot=True, fmt=".2f",
    cmap="coolwarm", center=0,
    linewidths=0.5, linecolor="white",
    vmin=-1, vmax=1,
    annot_kws={"size": 9}
)
ax.set_title("Cross-System Numeric Correlation Matrix — int_quality__orders_enriched")
plt.tight_layout()
plt.show()

# ── Highlight strongest correlations with defect_rate ─────────────────────
if "defect_rate" in corr_matrix.columns:
    dr_corr = (
        corr_matrix["defect_rate"]
        .drop("defect_rate")
        .abs()
        .sort_values(ascending=False)
    )
    print("\nCorrelations with defect_rate (absolute value, descending):")
    print(dr_corr.to_string())

---
## 4. Defect Rate Analysis

In [ ]:
# ── Analysis config ───────────────────────────────────────────────────────
# Minimum work orders per group to include in rate analysis.
# Groups below this threshold are shown but flagged as low-n.
MIN_GROUP_SIZE = 10

# Working dataframe — exclude rows with null defect_rate (uninspected)
df_dr = orders.dropna(subset=["defect_rate"]).copy()
print(f"Rows with valid defect_rate: {len(df_dr):,} of {len(orders):,}")

### 4.1 Overall Defect Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(df_dr["defect_rate"], bins=50, color=BRAND_BLUE,
             edgecolor="white", linewidth=0.3)
axes[0].axvline(df_dr["defect_rate"].mean(), color=WARN_AMBER,
                linestyle="--", linewidth=1.5,
                label=f"Mean ({df_dr['defect_rate'].mean():.3f})")
axes[0].axvline(df_dr["defect_rate"].median(), color=FAIL_RED,
                linestyle="--", linewidth=1.5,
                label=f"Median ({df_dr['defect_rate'].median():.3f})")
axes[0].set_title("Defect Rate Distribution")
axes[0].set_xlabel("Defect Rate")
axes[0].set_ylabel("Work Orders")
axes[0].legend()

# Box plot
axes[1].boxplot(df_dr["defect_rate"], vert=True, patch_artist=True,
                boxprops=dict(facecolor=BRAND_ACCENT, color=BRAND_BLUE),
                medianprops=dict(color=WARN_AMBER, linewidth=2),
                flierprops=dict(marker="o", markersize=3,
                                markerfacecolor=FAIL_RED, alpha=0.4))
axes[1].set_title("Defect Rate Box Plot")
axes[1].set_ylabel("Defect Rate")

plt.suptitle("Overall Defect Rate — int_quality__orders_enriched",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary stats
print("\nSummary statistics — defect_rate:")
print(df_dr["defect_rate"].describe().to_string())

### 4.2 Defect Rate by Dimension

One chart per pattern dimension. Each chart shows mean defect rate per group
with group size (n) annotated. Groups below `MIN_GROUP_SIZE` are flagged.

In [ ]:
def defect_rate_by_dimension(df, dim_col, title, highlight_col=None):
    """
    Bar chart of mean defect rate grouped by dim_col.
    Annotates each bar with group size. Highlights a specific group if provided.
    """
    grouped = (
        df.groupby(dim_col)["defect_rate"]
        .agg(mean_defect_rate="mean", n="count")
        .reset_index()
        .sort_values("mean_defect_rate", ascending=False)
    )
    grouped["low_n"] = grouped["n"] < MIN_GROUP_SIZE

    colors = []
    for _, row in grouped.iterrows():
        if row["low_n"]:
            colors.append(WARN_AMBER)
        elif highlight_col and row[dim_col] == highlight_col:
            colors.append(FAIL_RED)
        else:
            colors.append(BRAND_BLUE)

    fig, ax = plt.subplots(figsize=(max(8, len(grouped) * 1.2), 5))
    bars = ax.bar(range(len(grouped)), grouped["mean_defect_rate"],
                  color=colors, width=0.65)

    # Annotate bars with n
    for i, (_, row) in enumerate(grouped.iterrows()):
        label = f"n={row['n']:,}"
        if row["low_n"]:
            label += " ⚠"
        ax.text(i, row["mean_defect_rate"] + 0.001, label,
                ha="center", va="bottom", fontsize=9)

    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels(grouped[dim_col].astype(str), rotation=30, ha="right")
    ax.set_ylabel("Mean Defect Rate")
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.3f}"))

    # Overall mean reference line
    overall_mean = df["defect_rate"].mean()
    ax.axhline(overall_mean, color=WARN_AMBER, linestyle="--", linewidth=1.2,
               label=f"Overall mean ({overall_mean:.3f})")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n{title} — group summary:")
    display(grouped.reset_index(drop=True))

In [ ]:
# ── P1: Machine Type × Shift Code ─────────────────────────────────────────
df_p1 = df_dr.copy()
df_p1["machine_x_shift"] = df_p1["machine_type"] + " | " + df_p1["shift_code"].fillna("(no shift)")

defect_rate_by_dimension(
    df_p1, "machine_x_shift",
    "P1 — Mean Defect Rate by Machine Type × Shift",
    highlight_col="Press Brake | Shift B"
)

In [ ]:
# ── P2: Supplier ──────────────────────────────────────────────────────────
defect_rate_by_dimension(
    df_dr.dropna(subset=["supplier"]),
    "supplier",
    "P2 — Mean Defect Rate by Supplier",
)

In [ ]:
# ── P3: Complexity ────────────────────────────────────────────────────────
defect_rate_by_dimension(
    df_dr,
    "complexity",
    "P3 — Mean Defect Rate by Part Complexity",
)

In [ ]:
# ── P4: Welding Cert Mismatch ─────────────────────────────────────────────
df_p4 = df_dr[df_dr["requires_welding"] == True].copy()
df_p4["cert_status"] = df_p4["welding_cert_mismatch"].map(
    {True: "Cert Lapsed", False: "Cert Current"}
)

defect_rate_by_dimension(
    df_p4,
    "cert_status",
    "P4 — Mean Defect Rate by Welding Cert Status (Welding Jobs Only)",
)

---
## 5. Scrap Cost Attribution

Total and average scrap costs broken down by the key analytical dimensions.
Cross-references defect rate with cost impact — a high defect rate dimension
is not necessarily the highest cost dimension if the affected parts have low
material or labor values.

In [ ]:
# ── Time series config ────────────────────────────────────────────────────
# Numeric columns to plot as monthly time series against scrap_month.
SCRAP_TIME_SERIES_METRICS = ["total_scrap_cost", "quantity_scrapped"]

### 5.1 Total Scrap Cost by Dimension

In [ ]:
def scrap_cost_by_dimension(df, dim_col, title):
    """Bar chart of total and mean scrap cost grouped by dim_col."""
    grouped = (
        df.groupby(dim_col)["total_scrap_cost"]
        .agg(total_cost="sum", mean_cost="mean", n="count")
        .reset_index()
        .sort_values("total_cost", ascending=False)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total cost
    axes[0].bar(range(len(grouped)), grouped["total_cost"],
                color=BRAND_BLUE, width=0.65)
    axes[0].set_xticks(range(len(grouped)))
    axes[0].set_xticklabels(grouped[dim_col].astype(str), rotation=30, ha="right")
    axes[0].set_ylabel("Total Scrap Cost ($)")
    axes[0].set_title("Total Scrap Cost")
    axes[0].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )

    # Mean cost per event
    axes[1].bar(range(len(grouped)), grouped["mean_cost"],
                color=BRAND_ACCENT, width=0.65)
    axes[1].set_xticks(range(len(grouped)))
    axes[1].set_xticklabels(grouped[dim_col].astype(str), rotation=30, ha="right")
    axes[1].set_ylabel("Mean Cost per Scrap Event ($)")
    axes[1].set_title("Mean Scrap Cost per Event")
    axes[1].yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )

    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\n{title} — group summary:")
    display(grouped.reset_index(drop=True))

scrap_cost_by_dimension(scrap, "machine_type",  "Scrap Cost by Machine Type")
scrap_cost_by_dimension(scrap, "supplier",      "Scrap Cost by Supplier")
scrap_cost_by_dimension(scrap, "complexity",    "Scrap Cost by Complexity")
scrap_cost_by_dimension(scrap, "shift_code",    "Scrap Cost by Shift")

### 5.2 Defect Rate vs Scrap Cost Scatter

In [ ]:
# Join mean defect rate per work order to total scrap cost per work order.
# Each point is one work order.
scatter_df = (
    scrap.groupby("work_order_id")["total_scrap_cost"].sum()
    .reset_index()
    .merge(
        orders[["work_order_id", "defect_rate", "machine_type",
                  "complexity", "supplier"]].dropna(subset=["defect_rate"]),
        on="work_order_id", how="inner"
    )
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(scatter_df["defect_rate"], scatter_df["total_scrap_cost"],
           alpha=0.4, s=20, color=BRAND_BLUE)

# Trend line
x = scatter_df["defect_rate"].values
y = scatter_df["total_scrap_cost"].values
slope, intercept, r, _, _ = linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color=FAIL_RED,
        linewidth=1.5, linestyle="--", label=f"Trend (r={r:.2f})")

ax.set_xlabel("Defect Rate")
ax.set_ylabel("Total Scrap Cost per Work Order ($)")
ax.set_title("Defect Rate vs Total Scrap Cost — per Work Order")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()

print(f"Pearson r (defect_rate vs total_scrap_cost): {r:.3f}")

### 5.3 Monthly Scrap Cost Trend

In [ ]:
for metric in SCRAP_TIME_SERIES_METRICS:
    if metric not in scrap.columns:
        print(f"Column {metric} not found — skipping.")
        continue

    monthly = (
        scrap.groupby("scrap_month")[metric]
        .sum()
        .reset_index()
        .sort_values("scrap_month")
    )
    s   = monthly[metric]
    x_n = np.arange(len(s))
    slope, intercept, _, _, _ = linregress(x_n, s)
    trend = slope * x_n + intercept

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(monthly["scrap_month"].astype(str), s,
            color=BRAND_BLUE, linewidth=2, marker="o", markersize=4)
    ax.axhline(s.mean(), color=WARN_AMBER, linestyle="--", linewidth=1.2,
               label=f"Mean ({s.mean():,.1f})")
    ax.plot(monthly["scrap_month"].astype(str), trend,
            color=FAIL_RED, linewidth=1.5, linestyle="-.",
            label=f"Trend (slope: {slope:+.1f}/mo)")
    ax.set_title(f"Monthly {metric.replace('_', ' ').title()} — Scrap Events")
    ax.set_ylabel(metric.replace("_", " ").title())

    fmt = mticker.FuncFormatter(lambda x, _: f"${x:,.0f}") if "cost" in metric           else mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    ax.yaxis.set_major_formatter(fmt)

    xticks = monthly["scrap_month"].astype(str)[::6]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticks, rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n{metric} — monthly summary:")
    print(f"  Mean:   {s.mean():,.1f}")
    print(f"  Std:    {s.std():,.1f}")
    print(f"  Min:    {s.min():,.1f}  ({monthly.loc[s.idxmin(), 'scrap_month']})")
    print(f"  Max:    {s.max():,.1f}  ({monthly.loc[s.idxmax(), 'scrap_month']})")
    print(f"  Trend:  {slope:+.2f} per month")

### 5.4 Top Work Orders by Scrap Cost

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
TOP_N_ORDERS = 20

top_orders = (
    scrap.groupby("work_order_id")["total_scrap_cost"].sum()
    .reset_index()
    .sort_values("total_scrap_cost", ascending=False)
    .head(TOP_N_ORDERS)
    .merge(
        orders[["work_order_id", "part_number", "machine_type",
                  "shift_code", "operator_id", "supplier",
                  "complexity", "defect_rate", "welding_cert_mismatch"]],
        on="work_order_id", how="left"
    )
)

print(f"Top {TOP_N_ORDERS} work orders by total scrap cost:")
display(top_orders.reset_index(drop=True))

---
## 6. Operator & Certification Analysis

### 6.1 Defect Rate by Operator

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
MIN_JOBS_OPERATOR = 10  # minimum work orders for an operator to appear in chart

op_summary = (
    df_dr.groupby(["operator_id", "operator_name", "welding_cert_current",
                   "cert_level", "specialization"])["defect_rate"]
    .agg(mean_defect_rate="mean", n_jobs="count")
    .reset_index()
    .sort_values("mean_defect_rate", ascending=False)
)

op_plot = op_summary[op_summary["n_jobs"] >= MIN_JOBS_OPERATOR].copy()
op_plot["label"] = op_plot["operator_id"] + "\n" + op_plot["operator_name"].str.split().str[0]

colors = [
    FAIL_RED if not cert else BRAND_BLUE
    for cert in op_plot["welding_cert_current"]
]

fig, ax = plt.subplots(figsize=(max(10, len(op_plot) * 0.7), 5))
bars = ax.bar(range(len(op_plot)), op_plot["mean_defect_rate"],
              color=colors, width=0.65)

for i, (_, row) in enumerate(op_plot.iterrows()):
    ax.text(i, row["mean_defect_rate"] + 0.001, f"n={row['n_jobs']}",
            ha="center", va="bottom", fontsize=8)

ax.set_xticks(range(len(op_plot)))
ax.set_xticklabels(op_plot["label"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mean Defect Rate")
ax.set_title("Defect Rate by Operator (red = lapsed welding cert)")
ax.axhline(df_dr["defect_rate"].mean(), color=WARN_AMBER, linestyle="--",
           linewidth=1.2, label=f"Overall mean ({df_dr['defect_rate'].mean():.3f})")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.3f}"))
ax.legend()
plt.tight_layout()
plt.show()

print("\nOperator summary (all operators with sufficient job volume):")
display(op_plot.drop(columns=["label"]).reset_index(drop=True))

### 6.2 Cert Lapsed vs Current — Statistical Comparison

In [ ]:
# Two-sample t-test: defect rate for welding jobs with lapsed cert vs current cert.
# Restricted to welding jobs only (requires_welding = True).
welding_jobs = df_dr[df_dr["requires_welding"] == True].copy()

lapsed  = welding_jobs[welding_jobs["welding_cert_mismatch"] == True]["defect_rate"]
current = welding_jobs[welding_jobs["welding_cert_mismatch"] == False]["defect_rate"]

t_stat, p_val = ttest_ind(lapsed, current, equal_var=False)

print("Welding jobs only — defect rate by certification status:")
print(f"  Cert lapsed  — n={len(lapsed):,}  mean={lapsed.mean():.4f}  std={lapsed.std():.4f}")
print(f"  Cert current — n={len(current):,}  mean={current.mean():.4f}  std={current.std():.4f}")
print(f"\nWelch t-test: t={t_stat:.3f}  p={p_val:.4f}")
print(f"Result: {'Statistically significant (p < 0.05)' if p_val < 0.05 else 'Not significant at p < 0.05'}")

# ── Box plot comparison ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [current, lapsed],
    labels=["Cert Current", "Cert Lapsed"],
    patch_artist=True,
    boxprops=dict(facecolor=BRAND_ACCENT),
    medianprops=dict(color=WARN_AMBER, linewidth=2),
    flierprops=dict(marker="o", markersize=3, markerfacecolor=FAIL_RED, alpha=0.4)
)
ax.set_ylabel("Defect Rate")
ax.set_title("P4 — Defect Rate: Cert Current vs Cert Lapsed (Welding Jobs Only)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.3f}"))
plt.tight_layout()
plt.show()

### 6.3 OP007 Spotlight

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
SPOTLIGHT_OPERATOR = "OP007"

op_df  = df_dr[df_dr["operator_id"] == SPOTLIGHT_OPERATOR]
all_df = df_dr[df_dr["operator_id"] != SPOTLIGHT_OPERATOR]

if op_df.empty:
    print(f"No records found for operator {SPOTLIGHT_OPERATOR}.")
else:
    print(f"Operator {SPOTLIGHT_OPERATOR} summary:")
    print(f"  Total work orders:   {len(op_df):,}")
    print(f"  Mean defect rate:    {op_df['defect_rate'].mean():.4f}")
    print(f"  All others mean:     {all_df['defect_rate'].mean():.4f}")
    print(f"  Multiplier:          {op_df['defect_rate'].mean() / all_df['defect_rate'].mean():.2f}x")
    print(f"  Welding cert status: {op_df['welding_cert_current'].iloc[0]}")
    print(f"  Cert level:          {op_df['cert_level'].iloc[0]}")
    print(f"  Specialization:      {op_df['specialization'].iloc[0]}")

    print(f"\n{SPOTLIGHT_OPERATOR} — defect rate by machine type:")
    display(
        op_df.groupby("machine_type")["defect_rate"]
        .agg(mean_defect_rate="mean", n_jobs="count")
        .sort_values("mean_defect_rate", ascending=False)
        .reset_index()
    )